# Turo 5-Year Simulation

## Objective
Estimate the investment performance of buying a vehicle and renting it on
Turo over a 5-year horizon, with realistic revenue adjustments, cost
modelling, and standard financial metrics.

## Inputs
### Variable inputs
- Vehicle gross monthly income (before Turo fees)
- Vehicle purchase value
- Depreciation schedule (per-year rates) or a single flat annual rate

### Revenue adjustments (defaults)
- Turo platform fee rate: `25%`
- Occupancy rate: `75%`
- Seasonal income factors: optional 12-element monthly multiplier

### Cost parameters (defaults)
- Annual operating / maintenance cost (year 1): `$2,400`
- Maintenance cost growth rate: `5%` per year
- Annual insurance cost: `$2,400`
- Income tax rate on net cashflow: `22%`
- Sale transaction cost rate: `5%`

### Financing & other defaults
- APR: `4.99%`
- Sales tax: `10%`
- Down payment rate: `0%`
- Loan term: `60 months`
- Annual risk-free rate (discount / growth rate): `3.99%`
- Simulation horizon: `5 years`

## Method
1. Compute financed purchase cost (purchase value + sales tax - down payment).
2. Simulate month-by-month:
   - Gross income x occupancy x seasonal factor - Turo fee = net rental income.
   - Subtract escalating operating cost, insurance, and loan payment.
   - Apply income tax to positive monthly cashflow.
3. Depreciate the vehicle each year using a per-year schedule
   (front-loaded by default).
4. At the end of the horizon, sell the vehicle (minus transaction cost).
5. Compute:
   - **Present Value (PV)** discounted to initiation date
   - **Future Value (FV)** compounded to horizon end
   - **Return on Capital** = `PV / Loan PV - 1`
   - **Total Return of FV** = `FV / Total Loan Paid - 1`
   - **IRR** (annualised internal rate of return on monthly cashflows)
   - **Cash-on-Cash Return** (year-1 after-tax cashflow / initial capital)
   - **Break-even month** (cumulative cashflow turns positive, excl. sale)
   - **Payback period** (cumulative cashflow incl. sale turns positive)

## Outputs
- Yearly pandas DataFrame: income, fees, costs, insurance, loan, tax,
  cashflow, asset value
- Terminal sale value (net of transaction cost)
- PV, FV, Return on Capital, Total Return of FV, IRR, Cash-on-Cash,
  break-even, payback
- Visualisations: return-rate curves, income / expense breakdown,
  sensitivity heatmap, Monte Carlo histogram, multi-vehicle comparison

In [ ]:
# Turo 5-year vehicle investment simulation (enhanced)

import numpy as np
import pandas as pd
from scipy.optimize import brentq
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------------
# Defaults
# ---------------------------------------------------------------------------
# Monthly seasonal multipliers (Jan = index 0 ... Dec = index 11).
# > 1 means above-average demand; < 1 means below-average.
DEFAULT_SEASONAL_FACTORS = [
    0.70, 0.75, 0.85, 0.95, 1.10, 1.25,
    1.30, 1.25, 1.10, 0.95, 0.80, 0.70,
]

# Per-year depreciation rates (year 1 steepest, then flattens).
DEFAULT_DEPRECIATION_SCHEDULE = [0.20, 0.15, 0.12, 0.10, 0.08]


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------
def _monthly_payment(principal: float, annual_rate: float, months: int) -> float:
    """Standard fixed-rate amortised loan payment."""
    if months <= 0:
        return 0.0
    r = annual_rate / 12
    if r == 0:
        return principal / months
    return principal * (r * (1 + r) ** months) / ((1 + r) ** months - 1)


def _compute_irr(cashflows: list[float]) -> float:
    """Annualised IRR from a monthly cashflow vector (index 0 = month 0)."""
    def _npv(r):
        return sum(cf / (1 + r) ** t for t, cf in enumerate(cashflows))
    try:
        monthly = brentq(_npv, -0.99, 100.0, maxiter=2000)
        return (1 + monthly) ** 12 - 1
    except (ValueError, RuntimeError):
        return float("nan")


# ---------------------------------------------------------------------------
# Main simulation
# ---------------------------------------------------------------------------
def run_turo_simulation(
    vehicle_monthly_income: float,
    vehicle_purchase_value: float,
    # Depreciation (schedule takes priority over flat rate)
    annual_depreciation_rate: float = 0.15,
    depreciation_schedule: list[float] | None = None,
    # Revenue adjustments
    turo_fee_rate: float = 0.25,
    occupancy_rate: float = 0.75,
    seasonal_factors: list[float] | None = None,
    # Additional costs
    insurance_annual_cost: float = 2_400.0,
    income_tax_rate: float = 0.22,
    maintenance_growth_rate: float = 0.05,
    sale_cost_rate: float = 0.05,
    # Financing & other
    apr: float = 0.0499,
    sales_tax_rate: float = 0.10,
    annual_operating_cost: float = 2_400.0,
    loan_term_months: int = 60,
    down_payment_rate: float = 0.00,
    annual_risk_free_rate: float = 0.0399,
    years: int = 5,
    verbose: bool = True,
):
    """Run Turo vehicle-rental investment simulation with realistic modelling."""
    if years <= 0:
        raise ValueError("years must be > 0")
    if loan_term_months <= 0:
        raise ValueError("loan_term_months must be > 0")

    # Resolve depreciation schedule ----------------------------------------
    if depreciation_schedule is not None:
        dep = list(depreciation_schedule)
    else:
        dep = [annual_depreciation_rate] * years
    while len(dep) < years:
        dep.append(dep[-1])
    dep = dep[:years]

    # Resolve seasonal factors ---------------------------------------------
    sf = list(seasonal_factors) if seasonal_factors is not None else [1.0] * 12
    if len(sf) != 12:
        raise ValueError("seasonal_factors must have exactly 12 elements")

    # Financing setup -------------------------------------------------------
    total_purchase = vehicle_purchase_value * (1 + sales_tax_rate)
    down_payment = total_purchase * down_payment_rate
    loan_principal = total_purchase - down_payment
    pmt = _monthly_payment(loan_principal, apr, loan_term_months)
    monthly_r = annual_risk_free_rate / 12
    total_months = years * 12

    # Asset value at each year-end ------------------------------------------
    asset_vals = [vehicle_purchase_value]
    for y in range(years):
        asset_vals.append(asset_vals[-1] * (1 - dep[y]))
    net_sale = asset_vals[-1] * (1 - sale_cost_rate)
    sale_cost_amt = asset_vals[-1] * sale_cost_rate

    # Month-by-month simulation ---------------------------------------------
    balance = loan_principal
    monthly_cf = [-down_payment]  # month 0
    agg = {
        y: dict(
            gross_income=0.0, turo_fees=0.0, net_rental_income=0.0,
            operating_cost=0.0, insurance=0.0, loan_paid=0.0,
            interest=0.0, principal_paid=0.0,
            pre_tax_cashflow=0.0, tax=0.0, after_tax_cashflow=0.0,
        )
        for y in range(1, years + 1)
    }

    for m in range(1, total_months + 1):
        yi = (m - 1) // 12           # 0-based year index
        yr = yi + 1                  # 1-based year
        cm = (m - 1) % 12            # calendar month 0-11

        gross = vehicle_monthly_income * sf[cm] * occupancy_rate
        fee = gross * turo_fee_rate
        net_rent = gross - fee

        op = (annual_operating_cost * (1 + maintenance_growth_rate) ** yi) / 12
        ins = insurance_annual_cost / 12

        if balance > 1e-9:
            intr = balance * (apr / 12)
            ppaid = min(pmt - intr, balance)
            loan_pay = intr + ppaid
            balance -= ppaid
        else:
            intr = ppaid = loan_pay = 0.0

        pre_tax = net_rent - op - ins - loan_pay
        tax = max(0.0, pre_tax * income_tax_rate) if income_tax_rate > 0 else 0.0
        after_tax = pre_tax - tax

        terminal = net_sale if m == total_months else 0.0
        monthly_cf.append(after_tax + terminal)

        a = agg[yr]
        a["gross_income"] += gross
        a["turo_fees"] += fee
        a["net_rental_income"] += net_rent
        a["operating_cost"] += op
        a["insurance"] += ins
        a["loan_paid"] += loan_pay
        a["interest"] += intr
        a["principal_paid"] += ppaid
        a["pre_tax_cashflow"] += pre_tax
        a["tax"] += tax
        a["after_tax_cashflow"] += after_tax

    # Build yearly DataFrame ------------------------------------------------
    bal = loan_principal
    rows = []
    for yr in range(1, years + 1):
        a = agg[yr]
        bal -= a["principal_paid"]
        rows.append({
            "year": yr, **a,
            "asset_value": asset_vals[yr],
            "ending_loan_balance": max(0, bal),
        })
    df = pd.DataFrame(rows).set_index("year")

    # PV / FV ---------------------------------------------------------------
    pv = sum(cf / (1 + monthly_r) ** t for t, cf in enumerate(monthly_cf))
    fv = sum(
        cf * (1 + monthly_r) ** (total_months - t)
        for t, cf in enumerate(monthly_cf)
    )
    loan_pv = sum(pmt / (1 + monthly_r) ** t for t in range(1, loan_term_months + 1))
    roc = (pv / loan_pv - 1) if loan_pv > 1e-9 else float("nan")
    total_loan = df["loan_paid"].sum()
    tr_fv = (fv / total_loan - 1) if total_loan > 1e-9 else float("nan")

    # IRR -------------------------------------------------------------------
    irr = _compute_irr(monthly_cf)

    # Cash-on-Cash ----------------------------------------------------------
    coc = (
        df.loc[1, "after_tax_cashflow"] / down_payment
        if down_payment > 1e-9
        else float("nan")
    )

    # Break-even & Payback --------------------------------------------------
    cum = 0.0
    be_month = None
    for t, cf in enumerate(monthly_cf):
        cum += cf - (net_sale if t == total_months else 0)
        if cum > 0 and be_month is None:
            be_month = t

    cum = 0.0
    pb_month = None
    for t, cf in enumerate(monthly_cf):
        cum += cf
        if cum > 0 and pb_month is None:
            pb_month = t

    result = {
        "inputs": dict(
            vehicle_monthly_income=vehicle_monthly_income,
            vehicle_purchase_value=vehicle_purchase_value,
            depreciation_schedule=dep,
            turo_fee_rate=turo_fee_rate,
            occupancy_rate=occupancy_rate,
            seasonal_factors=sf,
            insurance_annual_cost=insurance_annual_cost,
            income_tax_rate=income_tax_rate,
            maintenance_growth_rate=maintenance_growth_rate,
            sale_cost_rate=sale_cost_rate,
            apr=apr,
            sales_tax_rate=sales_tax_rate,
            annual_operating_cost=annual_operating_cost,
            loan_term_months=loan_term_months,
            down_payment_rate=down_payment_rate,
            annual_risk_free_rate=annual_risk_free_rate,
            years=years,
        ),
        "yearly_df": df,
        "yearly_rows": rows,
        "net_sale_value": net_sale,
        "sale_cost": sale_cost_amt,
        "cashflows": monthly_cf,
        "present_value": pv,
        "future_value": fv,
        "return_on_capital": roc,
        "total_return_future_value": tr_fv,
        "irr": irr,
        "cash_on_cash_return": coc,
        "breakeven_month": be_month,
        "payback_month": pb_month,
    }

    if verbose:
        _print_report(result)
    return result


def _print_report(res):
    """Pretty-print simulation results."""
    inp = res["inputs"]
    df = res["yearly_df"]
    yrs = inp["years"]
    dep_str = ", ".join(f"{d:.0%}" for d in inp["depreciation_schedule"])
    has_season = any(s != 1.0 for s in inp["seasonal_factors"])

    print("=" * 65)
    print("  INPUT ASSUMPTIONS")
    print("=" * 65)
    print(f"  Gross monthly income      ${inp['vehicle_monthly_income']:>10,.2f}")
    print(f"  Vehicle purchase value     ${inp['vehicle_purchase_value']:>10,.2f}")
    print(f"  Turo fee rate              {inp['turo_fee_rate']:>10.0%}")
    print(f"  Occupancy rate             {inp['occupancy_rate']:>10.0%}")
    print(f"  Depreciation schedule      [{dep_str}]")
    print(f"  Insurance (annual)         ${inp['insurance_annual_cost']:>10,.2f}")
    print(f"  Operating cost (yr 1)      ${inp['annual_operating_cost']:>10,.2f}")
    print(f"  Maintenance growth         {inp['maintenance_growth_rate']:>10.0%}/yr")
    print(f"  APR                        {inp['apr']:>10.2%}")
    print(f"  Sales tax                  {inp['sales_tax_rate']:>10.0%}")
    print(f"  Loan term                  {inp['loan_term_months']:>10} months")
    print(f"  Down payment               {inp['down_payment_rate']:>10.0%}")
    print(f"  Income tax rate            {inp['income_tax_rate']:>10.0%}")
    print(f"  Sale cost rate             {inp['sale_cost_rate']:>10.0%}")
    print(f"  Risk-free rate             {inp['annual_risk_free_rate']:>10.2%}")
    print(f"  Seasonal income            {'Yes' if has_season else 'No (flat)':>10}")
    print()
    print("=" * 65)
    print("  YEARLY RESULTS")
    print("=" * 65)
    cols = [
        "net_rental_income", "operating_cost", "insurance",
        "loan_paid", "tax", "after_tax_cashflow",
        "asset_value", "ending_loan_balance",
    ]
    fmt = df[cols].copy()
    for c in fmt.columns:
        fmt[c] = fmt[c].map(lambda x: f"${x:,.0f}")
    display(fmt)
    print()
    print("=" * 65)
    print(f"  {yrs}-YEAR SUMMARY")
    print("=" * 65)
    scr = inp["sale_cost_rate"]
    print(f"  Net sale value (after {scr:.0%} cost)   ${res['net_sale_value']:>12,.2f}")
    print(f"  Present Value (PV)               ${res['present_value']:>12,.2f}")
    print(f"  Future Value (FV)                ${res['future_value']:>12,.2f}")
    print(f"  Return on Capital (PV/Loan PV)   {res['return_on_capital']:>12.2%}")
    print(f"  Total Return (FV/Loan Paid)      {res['total_return_future_value']:>12.2%}")
    print(f"  IRR (annualised)                 {res['irr']:>12.2%}")
    coc = res["cash_on_cash_return"]
    coc_s = f"{coc:.2%}" if not np.isnan(coc) else "N/A (0% down)"
    print(f"  Cash-on-Cash (Year 1)            {coc_s:>12}")
    be = res["breakeven_month"]
    print(f"  Break-even (excl. sale)          {'Mo ' + str(be) if be else 'Never':>12}")
    pb = res["payback_month"]
    print(f"  Payback (incl. sale)             {'Mo ' + str(pb) if pb else 'Never':>12}")


# === Run with defaults ===
result = run_turo_simulation(
    vehicle_monthly_income=1800.0,
    vehicle_purchase_value=32000.0,
    depreciation_schedule=DEFAULT_DEPRECIATION_SCHEDULE,
    seasonal_factors=DEFAULT_SEASONAL_FACTORS,
)

In [ ]:
def plot_analysis(sim_result: dict):
    """Return-rate curves and income / expense breakdown."""
    rows = sim_result["yearly_rows"]
    inp = sim_result["inputs"]
    df = sim_result["yearly_df"]

    purchase_tax = inp["vehicle_purchase_value"] * (1 + inp["sales_tax_rate"])
    dp = purchase_tax * inp["down_payment_rate"]

    # 1. Cumulative & YoY return rate (sell-at-year-end assumption) ---------
    yrs, rr = [], []
    cum_cf = cum_loan = 0.0
    for r in rows:
        cum_cf += r["after_tax_cashflow"]
        cum_loan += r["loan_paid"]
        invested = dp + cum_loan
        equity = r["asset_value"] - r["ending_loan_balance"]
        rate = (cum_cf + equity) / invested - 1 if invested > 1e-9 else float("nan")
        yrs.append(r["year"])
        rr.append(rate)

    yoy = []
    for i, r in enumerate(rr):
        if i == 0:
            yoy.append(r)
        else:
            yoy.append(
                ((1 + r) / (1 + rr[i - 1]) - 1)
                if abs(1 + rr[i - 1]) > 1e-9
                else float("nan")
            )

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    axes[0].plot(yrs, [x * 100 for x in rr], marker="o")
    axes[0].axhline(0, ls="--", lw=1)
    axes[0].set(title="Total Return Rate by Year", xlabel="Year", ylabel="%")
    axes[0].set_xticks(yrs)
    axes[0].grid(alpha=0.3)

    axes[1].plot(yrs, [x * 100 for x in yoy], marker="o", color="tab:orange")
    axes[1].axhline(0, ls="--", lw=1)
    axes[1].set(title="Year-over-Year Return Rate", xlabel="Year", ylabel="%")
    axes[1].set_xticks(yrs)
    axes[1].grid(alpha=0.3)

    # 2. Income vs expense breakdown ----------------------------------------
    x = np.arange(1, len(rows) + 1)
    w = 0.35
    axes[2].bar(
        x - w / 2, df["net_rental_income"].values, w,
        label="Net rental income", color="tab:green",
    )
    bottom = np.zeros(len(rows))
    for vals, lbl, clr in [
        (df["operating_cost"].values, "Operating", "tab:red"),
        (df["insurance"].values, "Insurance", "tab:purple"),
        (df["loan_paid"].values, "Loan payment", "tab:blue"),
        (df["tax"].values, "Tax", "tab:gray"),
    ]:
        axes[2].bar(x + w / 2, vals, w, bottom=bottom, label=lbl, color=clr)
        bottom += vals
    axes[2].set(title="Income vs Expenses", xlabel="Year", ylabel="$")
    axes[2].set_xticks(x)
    axes[2].legend(fontsize=8)
    axes[2].grid(alpha=0.3, axis="y")

    fig.suptitle("Sell-at-Year-End Analysis", y=1.02, fontsize=14)
    plt.tight_layout()
    plt.show()

    for y, tr, yr in zip(yrs, rr, yoy):
        print(f"Year {y}: Total={tr:.2%}, YoY={yr:.2%}")
    best_y, best_r = max(zip(yrs, rr), key=lambda t: t[1])
    print(f"\nRecommended sell year (max total return): Year {best_y} ({best_r:.2%})")
    return yrs, rr, best_y, best_r


plot_analysis(result)

In [ ]:
def plot_sensitivity(base_purchase=32000,
                     income_range=None, dep_range=None):
    """Heatmap of IRR across monthly-income x depreciation-rate grid."""
    if income_range is None:
        income_range = np.arange(1000, 3001, 250)
    if dep_range is None:
        dep_range = np.arange(0.08, 0.26, 0.02)

    grid = np.zeros((len(dep_range), len(income_range)))
    for i, d in enumerate(dep_range):
        for j, inc in enumerate(income_range):
            r = run_turo_simulation(
                vehicle_monthly_income=inc,
                vehicle_purchase_value=base_purchase,
                annual_depreciation_rate=d,
                depreciation_schedule=None,
                seasonal_factors=DEFAULT_SEASONAL_FACTORS,
                verbose=False,
            )
            grid[i, j] = r["irr"] * 100

    fig, ax = plt.subplots(figsize=(10, 6))
    im = ax.imshow(
        grid, aspect="auto", origin="lower",
        extent=[income_range[0], income_range[-1],
                dep_range[0] * 100, dep_range[-1] * 100],
        cmap="RdYlGn",
    )
    fig.colorbar(im, ax=ax, label="IRR (%)")
    ax.set(
        xlabel="Monthly Gross Income ($)",
        ylabel="Annual Depreciation Rate (%)",
        title=f"IRR Sensitivity (vehicle=${base_purchase:,.0f})",
    )
    plt.tight_layout()
    plt.show()


plot_sensitivity()

In [ ]:
def run_monte_carlo(n=2000, base_income=1800, income_std=300,
                    base_purchase=32000,
                    dep_mean=0.15, dep_std=0.04,
                    occ_mean=0.75, occ_std=0.10):
    """Monte Carlo simulation of IRR distribution."""
    rng = np.random.default_rng(42)
    irrs = []
    for _ in range(n):
        inc = max(500, rng.normal(base_income, income_std))
        dep = float(np.clip(rng.normal(dep_mean, dep_std), 0.03, 0.40))
        occ = float(np.clip(rng.normal(occ_mean, occ_std), 0.30, 1.00))
        r = run_turo_simulation(
            vehicle_monthly_income=inc,
            vehicle_purchase_value=base_purchase,
            annual_depreciation_rate=dep,
            depreciation_schedule=None,
            occupancy_rate=occ,
            seasonal_factors=DEFAULT_SEASONAL_FACTORS,
            verbose=False,
        )
        irrs.append(r["irr"] * 100)

    irrs = np.array(irrs)
    irrs = irrs[np.isfinite(irrs)]

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.hist(irrs, bins=50, edgecolor="k", alpha=0.7)
    med = float(np.median(irrs))
    p5 = float(np.percentile(irrs, 5))
    p95 = float(np.percentile(irrs, 95))
    for v, lbl, clr in [
        (med, f"Median {med:.1f}%", "blue"),
        (p5, f"5th pctl {p5:.1f}%", "red"),
        (p95, f"95th pctl {p95:.1f}%", "green"),
    ]:
        ax.axvline(v, ls="--", color=clr, lw=2, label=lbl)
    ax.set(
        xlabel="IRR (%)", ylabel="Frequency",
        title=f"Monte Carlo IRR Distribution (n={len(irrs)})",
    )
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

    print(f"Median IRR:      {med:>8.2f}%")
    print(f"5th percentile:  {p5:>8.2f}%")
    print(f"95th percentile: {p95:>8.2f}%")
    print(f"P(IRR > 0):      {(irrs > 0).mean():>8.1%}")


run_monte_carlo()

In [ ]:
def compare_vehicles(vehicles: list[dict]):
    """Compare multiple vehicles on key metrics.

    Each dict: {"name": str, "income": float, "purchase": float, "depreciation": float}.
    """
    records = []
    raw = []
    for v in vehicles:
        r = run_turo_simulation(
            vehicle_monthly_income=v["income"],
            vehicle_purchase_value=v["purchase"],
            annual_depreciation_rate=v["depreciation"],
            depreciation_schedule=None,
            seasonal_factors=DEFAULT_SEASONAL_FACTORS,
            verbose=False,
        )
        raw.append(r)
        pb = r["payback_month"]
        records.append({
            "Vehicle": v["name"],
            "Purchase": f"${v['purchase']:,.0f}",
            "Mo. Income": f"${v['income']:,.0f}",
            "Depr.": f"{v['depreciation']:.0%}",
            "IRR": f"{r['irr']:.2%}",
            "PV": f"${r['present_value']:,.0f}",
            "FV": f"${r['future_value']:,.0f}",
            "Payback": f"Mo {pb}" if pb else "Never",
        })

    display(pd.DataFrame(records))

    # IRR bar chart
    fig, ax = plt.subplots(figsize=(8, 4))
    names = [v["name"] for v in vehicles]
    irrs = [r["irr"] * 100 for r in raw]
    colors = ["tab:green" if x > 0 else "tab:red" for x in irrs]
    ax.barh(names, irrs, color=colors)
    ax.axvline(0, color="k", lw=1)
    ax.set(xlabel="IRR (%)", title="Vehicle Comparison - IRR")
    ax.grid(alpha=0.3, axis="x")
    plt.tight_layout()
    plt.show()

    # Opportunity cost vs S&P 500 benchmark
    benchmark = 0.10  # ~10% avg annual
    monthly_bm = benchmark / 12
    print("\nOpportunity cost vs S&P 500 (~10%/yr):")
    for v, r in zip(vehicles, raw):
        total_paid = r["yearly_df"]["loan_paid"].sum()
        tm = r["inputs"]["years"] * 12
        sp_fv = sum(
            (total_paid / tm) * (1 + monthly_bm) ** (tm - m)
            for m in range(1, tm + 1)
        )
        delta = r["future_value"] - sp_fv
        arrow = "+" if delta > 0 else "-"
        print(
            f"  {v['name']:20s}  Turo FV=${r['future_value']:>10,.0f}"
            f"  S&P FV~${sp_fv:>10,.0f}"
            f"  {arrow}${abs(delta):,.0f}"
        )


vehicles = [
    {"name": "Toyota Camry",  "income": 1200, "purchase": 25000, "depreciation": 0.12},
    {"name": "Tesla Model 3", "income": 1800, "purchase": 32000, "depreciation": 0.15},
    {"name": "BMW X5",        "income": 2500, "purchase": 50000, "depreciation": 0.20},
]
compare_vehicles(vehicles)

In [ ]:
# Final decision summary
_, _, best_year, best_rate = plot_analysis(result)

benchmark = 0.10
total_paid = result["yearly_df"]["loan_paid"].sum()
yrs = result["inputs"]["years"]
tm = yrs * 12
monthly_bm = benchmark / 12
sp_fv = sum(
    (total_paid / tm) * (1 + monthly_bm) ** (tm - m)
    for m in range(1, tm + 1)
)

print(f"\n{'=' * 65}")
print(f"  DECISION")
print(f"{'=' * 65}")
print(f"  Optimal sell year:  Year {best_year} (return = {best_rate:.2%})")
print(f"  IRR:                {result['irr']:.2%}")
print(f"  Turo FV:            ${result['future_value']:,.0f}")
print(f"  S&P 500 FV:         ${sp_fv:,.0f} (same payments at ~{benchmark:.0%}/yr)")
delta = result["future_value"] - sp_fv
verb = "outperforms" if delta > 0 else "underperforms"
print(f"  Turo {verb} S&P 500 by ${abs(delta):,.0f}")